# 09. Preprocessing (전처리)

## 목적
슬라이딩 윈도우 통합 데이터의 결측치 처리

## 입력
- `sliding_window_merged.csv`: 슬라이딩 윈도우 통합 데이터

## 출력
- `preprocessed.csv`: 결측치 처리된 데이터

## 전처리 범위
- [x] Forward fill (환자별 시간순)
- [x] Median imputation (남은 결측치)
- [ ] Feature Engineering → 다음 단계에서

In [58]:
import pandas as pd
import numpy as np
import os

INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

print("=== 11. Preprocessing 시작 ===")

=== 11. Preprocessing 시작 ===


## Step 1: 데이터 로드

In [59]:
print("Step 1: 데이터 로드")

df = pd.read_csv(
    os.path.join(INPUT_DIR, 'sliding_window_merged.csv'),
    parse_dates=['observation_start', 'observation_end']
)

print(f"✓ 데이터 로드 완료: {len(df):,} rows")

# 피처 컬럼 정의
vital_cols = ['hr', 'rr', 'spo2', 'temp', 'sbp', 'dbp', 'mbp']
lab_cols = ['sao2', 'ph', 'lactate', 'creatinine', 'bilirubin', 'wbc', 'platelets', 'potassium', 'sodium']
gcs_cols = ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']
urine_cols = ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag']

feature_cols = vital_cols + lab_cols + gcs_cols + urine_cols

Step 1: 데이터 로드
✓ 데이터 로드 완료: 941,817 rows


## Step 2: 결측치 현황 확인

In [60]:
print("\nStep 2: 결측치 현황 (처리 전)")

print("\n=== 피처별 결측치 비율 ===")
for col in feature_cols:
    if col in df.columns:
        missing = df[col].isna().mean() * 100
        print(f"  {col}: {missing:.1f}%")


Step 2: 결측치 현황 (처리 전)

=== 피처별 결측치 비율 ===
  hr: 0.9%
  rr: 2.1%
  spo2: 1.2%
  temp: 4.3%
  sbp: 2.5%
  dbp: 2.5%
  mbp: 2.7%
  sao2: 91.5%
  ph: 68.4%
  lactate: 71.6%
  creatinine: 8.7%
  bilirubin: 65.7%
  wbc: 10.4%
  platelets: 10.2%
  potassium: 8.8%
  sodium: 9.0%
  gcs_eye: 9.5%
  gcs_verbal: 9.5%
  gcs_motor: 9.7%
  gcs_total: 9.3%
  urine_ml_6h: 23.9%
  urine_ml_kg_hr_avg: 23.9%
  oliguria_flag: 23.9%


## Step 3: Vital Signs 결측치 처리

* 대상: hr, rr, spo2, temp, sbp, dbp, mbp
    * 결측률: 1~4% (매우 낮음)

* 전략: FFill → Median
    * ICU에서 거의 hourly 모니터링 → 결측 = 센서 일시 탈착 수준
    * 직전 값으로 채우는 것이 임상적으로 합리적
    * 결측률 낮아서 복잡한 처리 불필요

In [61]:
print("Step 3: Vital Signs 결측치 처리")

# 환자별, 시간순 정렬 (FFill 전 필수)
df = df.sort_values(['stay_id', 'observation_hour']).reset_index(drop=True)

# --- 3-1: HR, RR, SpO2, SBP, DBP, MBP (결측률 1~3%) ---
# FFill → Median (단순 처리)
vital_simple = ['hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp']

for col in vital_simple:
    before_missing = df[col].isna().sum()
    
    # FFill (환자별)
    df[col] = df.groupby('stay_id')[col].ffill()
    after_ffill = df[col].isna().sum()
    
    # 남은 결측 → 전체 Median
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
    print(f"  {col}: {before_missing:,} → FFill → {after_ffill:,} → Median({median_val:.1f}) → 0")

# --- 3-2: Temp (결측률 4.3%) ---
# FFill with limit=6 → Median
# 이유: 체온은 천천히 변하지만, 6시간 넘으면 신뢰도 하락
col = 'temp'
before_missing = df[col].isna().sum()

df[col] = df.groupby('stay_id')[col].ffill(limit=6)  # 최대 6시간까지만 forward fill
after_ffill = df[col].isna().sum()

median_val = df[col].median()
df[col] = df[col].fillna(median_val)

print(f"  {col}: {before_missing:,} → FFill(limit=6) → {after_ffill:,} → Median({median_val:.1f}) → 0")

print("✓ Vital Signs 처리 완료\n")

Step 3: Vital Signs 결측치 처리
  hr: 8,088 → FFill → 4,789 → Median(81.3) → 0
  rr: 19,447 → FFill → 8,534 → Median(18.7) → 0
  spo2: 11,190 → FFill → 5,191 → Median(96.2) → 0
  sbp: 23,299 → FFill → 9,065 → Median(121.5) → 0
  dbp: 23,312 → FFill → 9,067 → Median(66.2) → 0
  mbp: 25,490 → FFill → 9,384 → Median(81.3) → 0
  temp: 40,704 → FFill(limit=6) → 13,857 → Median(36.8) → 0
✓ Vital Signs 처리 완료



## Step 4: Lab Values (Lactate 제외) 처리

* 대상: creatinine, wbc, platelets, potassium, sodium
    * 결측률: 8~10%


- creatinine, wbc, platelets: FFill(limit=24) → Median
    - Lab은 의사 오더 기반 비정기 검사 → 일단위로 유효
    - 신장/혈액 지표는 24시간 내 값 신뢰 가능
- potassium, sodium: FFill(limit=12) → Median
    - 전해질(K, Na)은 변동 빠름 → 12시간으로 제한

In [62]:
print("Step 4: Lab Values 결측치 처리 (Lactate 제외)")

# --- 4-1: Creatinine, WBC, Platelets (결측률 8~10%) ---
# FFill(limit=24) → Median
lab_24h = ['creatinine', 'wbc', 'platelets']

for col in lab_24h:
    before_missing = df[col].isna().sum()
    
    df[col] = df.groupby('stay_id')[col].ffill(limit=24)
    after_ffill = df[col].isna().sum()
    
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
    print(f"  {col}: {before_missing:,} → FFill(limit=24) → {after_ffill:,} → Median({median_val:.2f}) → 0")

# --- 4-2: Potassium, Sodium (결측률 ~9%) ---
# FFill(limit=12) → Median
# 이유: 전해질은 변동 가능성 높음 → 12시간으로 제한
lab_12h = ['potassium', 'sodium']

for col in lab_12h:
    before_missing = df[col].isna().sum()
    
    df[col] = df.groupby('stay_id')[col].ffill(limit=12)
    after_ffill = df[col].isna().sum()
    
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
    print(f"  {col}: {before_missing:,} → FFill(limit=12) → {after_ffill:,} → Median({median_val:.2f}) → 0")

print("✓ Lab Values (Lactate 제외) 처리 완료\n")

Step 4: Lab Values 결측치 처리 (Lactate 제외)
  creatinine: 82,132 → FFill(limit=24) → 57,047 → Median(0.90) → 0
  wbc: 97,530 → FFill(limit=24) → 67,486 → Median(9.90) → 0
  platelets: 96,293 → FFill(limit=24) → 66,204 → Median(195.00) → 0
  potassium: 82,485 → FFill(limit=12) → 61,563 → Median(4.00) → 0
  sodium: 84,318 → FFill(limit=12) → 63,628 → Median(139.00) → 0
✓ Lab Values (Lactate 제외) 처리 완료



## Step 5: Lactate 결측치 처리

* 결측률: 71.6% (매우 높음)
* 전략: Missing Indicator + 조건부 Default
* 핵심 인사이트:
    - 71% 결측 = 대부분 "측정 안 함" = 의사가 필요 없다 판단
    - 측정 여부 자체가 강력한 예측 피처
        → 응급/중증 환자일수록 Lactate 측정함
        → 결측 = 상대적 안정 상태일 가능성
    - 단순 Median 채우면 실제 위험 환자와 섞여서 정보 손실
* 처리 로직:
    1) lactate_missing 플래그 생성 (0=측정함, 1=측정안함)
    2) 결측 시 조건부 Default:
        - MAP < 65 (저혈압) → 2.5 mmol/L (경도 상승 가정)
        - 그 외 → 1.3 mmol/L (정상 상한)
    3) 측정된 값은 그대로 유지

* SHAP 해석 시:
    - lactate_missing=1 & lactate_filled=1.3 → "측정 안 함, 안정 추정"
    - lactate_missing=0 & lactate_filled=4.5 → "측정함, 실제 높음 → 위험!"

In [63]:
print("Step 5: Lactate 결측치 처리 (특별 처리)")

# --- 5-1: Missing Indicator 생성 ---
df['lactate_missing'] = df['lactate'].isna().astype(int)
print(f"  lactate_missing 생성: 1(결측)={df['lactate_missing'].sum():,}, 0(측정)={(1-df['lactate_missing']).sum():,}")

# --- 5-2: 조건부 Default 적용 ---
# 정상 Lactate 범위: 0.5 ~ 2.0 mmol/L
LACTATE_NORMAL = 1.3      # 정상 상한 (보수적)
LACTATE_ELEVATED = 2.5    # 경도 상승 (저혈압 시)
MAP_THRESHOLD = 65        # 저혈압 기준

def impute_lactate(row):
    """
    Lactate 조건부 Imputation
    - 측정값 있으면 그대로 사용
    - 결측 시: MAP 기준으로 다르게 추정
    """
    if pd.notna(row['lactate']):
        return row['lactate']
    
    # 저혈압 (MAP < 65) → 조직 관류 저하 → Lactate 상승 추정
    if row['mbp'] < MAP_THRESHOLD:
        return LACTATE_ELEVATED
    else:
        return LACTATE_NORMAL

df['lactate'] = df.apply(impute_lactate, axis=1)

print(f"  조건부 Default 적용:")
print(f"    - MAP < {MAP_THRESHOLD} → {LACTATE_ELEVATED} mmol/L")
print(f"    - MAP >= {MAP_THRESHOLD} → {LACTATE_NORMAL} mmol/L")
print(f"  처리 후 결측: {df['lactate'].isna().sum()}")

# --- 5-3: 분포 확인 ---
print(f"\n  Lactate 분포 (처리 후):")
print(f"    - Mean: {df['lactate'].mean():.2f}")
print(f"    - Median: {df['lactate'].median():.2f}")
print(f"    - Min: {df['lactate'].min():.2f}, Max: {df['lactate'].max():.2f}")

print("✓ Lactate 처리 완료\n")

Step 5: Lactate 결측치 처리 (특별 처리)
  lactate_missing 생성: 1(결측)=674,429, 0(측정)=267,388
  조건부 Default 적용:
    - MAP < 65 → 2.5 mmol/L
    - MAP >= 65 → 1.3 mmol/L
  처리 후 결측: 0

  Lactate 분포 (처리 후):
    - Mean: 1.52
    - Median: 1.30
    - Min: 0.20, Max: 24.00
✓ Lactate 처리 완료



## Step 6: GCS 결측치 처리

* 대상: gcs_eye, gcs_verbal, gcs_motor, gcs_total
* 결측률: ~9.5%

1. 메인 변수: latest_gcs (window 내 가장 최근 관측값)
    - min_gcs, mean_gcs는 사용 안 함 (과거 과도 반영 방지)

2. 결측치 처리 원칙 (3가지 동시 적용):
    
    ① 결측 기본 유지: 무리한 대체로 정보 왜곡 방지
    - GCS 결측 ≠ 무작위. sedation/intubation 등 중증 상태와 연관
    
    ② Forward Fill: 시간 연속성 유지, 모델 입력 안정성
    
    ③ 결측 플래그: gcs_missing_flag로 중증 신호 보존

3. 출력 변수:
    - gcs_eye, gcs_verbal, gcs_motor: FFill 적용된 컴포넌트
    - gcs_total: 재계산 (Eye + Verbal + Motor)
    - gcs_original: FFill 전 원본 (결측 보존)
    - gcs_missing_flag: 결측 여부 (1=결측 있었음, 0=관측됨)

In [64]:
print("Step 6: GCS 결측치 처리 (팀 문서 기준)")

# --- 6-1: 원본 보존 (결측 그대로 유지 버전) ---
# 이유: GCS 결측은 sedation/intubation 등 중증 상태 신호일 수 있음
print("\n  6-1: 원본 결측 보존")

df['gcs_eye_original'] = df['gcs_eye'].copy()
df['gcs_verbal_original'] = df['gcs_verbal'].copy()
df['gcs_motor_original'] = df['gcs_motor'].copy()

# gcs_total_original: 컴포넌트 중 하나라도 NaN이면 NaN
df['gcs_total_original'] = df['gcs_eye_original'] + df['gcs_verbal_original'] + df['gcs_motor_original']

original_missing = df['gcs_total_original'].isna().sum()
print(f"    gcs_total_original 결측: {original_missing:,}건 ({original_missing/len(df)*100:.1f}%)")

# --- 6-2: 결측 플래그 생성 ---
# 이유: 결측 여부 자체가 중증 신호 (sedation, intubation 등)
print("\n  6-2: 결측 플래그 생성")

df['gcs_missing_flag'] = df['gcs_total_original'].isna().astype(int)

missing_count = df['gcs_missing_flag'].sum()
print(f"    gcs_missing_flag=1 (결측): {missing_count:,}건 ({missing_count/len(df)*100:.1f}%)")
print(f"    gcs_missing_flag=0 (관측): {len(df)-missing_count:,}건")

# --- 6-3: Forward Fill 적용 (메인 변수용) ---
# 이유: 시간 연속성 유지, 모델 입력 안정성 확보
# limit 없음: 팀 문서에서 "직전 시점의 가장 최근 관측값으로 ffill"
print("\n  6-3: Forward Fill 적용")

gcs_components = ['gcs_eye', 'gcs_verbal', 'gcs_motor']

for col in gcs_components:
    before_missing = df[col].isna().sum()
    
    # FFill (환자별, limit 없음)
    df[col] = df.groupby('stay_id')[col].ffill()
    after_ffill = df[col].isna().sum()
    
    print(f"    {col}: {before_missing:,} → {after_ffill:,} (FFill)")

# --- 6-4: FFill 후에도 남은 결측 처리 ---
# 이유: 환자의 첫 윈도우에 GCS 기록 없는 경우
# 전략: Median으로 채움 (모델 입력 안정성)
print("\n  6-4: 잔여 결측 Median 처리")

for col in gcs_components:
    remaining = df[col].isna().sum()
    if remaining > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"    {col}: {remaining:,}건 → Median({median_val:.1f})")

# --- 6-5: gcs_total 재계산 ---
# 이유: gcs_raw에서 NaN을 0으로 처리해서 잘못된 값 있음 (예: 2, 1)
# FFill 완료된 컴포넌트로 재계산해야 정확
print("\n  6-5: gcs_total 재계산")

before_error = (df['gcs_total'] < 3).sum()
print(f"    재계산 전 GCS<3: {before_error:,}건")

df['gcs_total'] = df['gcs_eye'] + df['gcs_verbal'] + df['gcs_motor']

# 범위 클리핑 (3~15)
df['gcs_total'] = df['gcs_total'].clip(lower=3, upper=15)

print(f"    재계산 후 범위: {df['gcs_total'].min():.0f} ~ {df['gcs_total'].max():.0f}")

# --- 6-6: 최종 확인 ---
print("\n=== GCS 변수 최종 구성 ===")
print("  [메인 변수 - FFill 적용]")
print(f"    gcs_eye, gcs_verbal, gcs_motor: 결측 0건")
print(f"    gcs_total: 범위 3~15, 결측 0건")
print("  [보조 변수 - 원본 보존]")
print(f"    gcs_*_original: 결측 {original_missing:,}건 보존")
print("  [플래그]")
print(f"    gcs_missing_flag: 결측={missing_count:,}, 관측={len(df)-missing_count:,}")

print("\n✓ GCS 처리 완료")

Step 6: GCS 결측치 처리 (팀 문서 기준)

  6-1: 원본 결측 보존
    gcs_total_original 결측: 93,615건 (9.9%)

  6-2: 결측 플래그 생성
    gcs_missing_flag=1 (결측): 93,615건 (9.9%)
    gcs_missing_flag=0 (관측): 848,202건

  6-3: Forward Fill 적용
    gcs_eye: 89,650 → 9,910 (FFill)
    gcs_verbal: 89,841 → 10,056 (FFill)
    gcs_motor: 91,630 → 10,351 (FFill)

  6-4: 잔여 결측 Median 처리
    gcs_eye: 9,910건 → Median(4.0)
    gcs_verbal: 10,056건 → Median(5.0)
    gcs_motor: 10,351건 → Median(6.0)

  6-5: gcs_total 재계산
    재계산 전 GCS<3: 393건
    재계산 후 범위: 3 ~ 15

=== GCS 변수 최종 구성 ===
  [메인 변수 - FFill 적용]
    gcs_eye, gcs_verbal, gcs_motor: 결측 0건
    gcs_total: 범위 3~15, 결측 0건
  [보조 변수 - 원본 보존]
    gcs_*_original: 결측 93,615건 보존
  [플래그]
    gcs_missing_flag: 결측=93,615, 관측=848,202

✓ GCS 처리 완료


## Step 7: Urine Output 처리

* 대상: urine_ml_6h, urine_ml_kg_hr_avg, oliguria_flag
* 결측률: ~24%
* 전략: Missing Indicator + 0/보수적 값 채움

1. 결측 의미:
    - 24% 결측은 무시 못 할 수준
    - 원인: (1) Foley 카테터 없음 (2) 기록 누락 (3) 실제 무뇨/핍뇨
    - 결측 자체가 임상적 신호일 수 있음
2. 처리 원칙 (GCS와 동일 구조):
    1. 원본 보존: 결측 그대로 유지한 버전
    2. 결측 플래그: urine_missing_flag로 신호 보존
    3. 보수적 대체: 메인 변수는 0으로 채움 (위험 가정)
3. 이상치 클리핑:
    - urine_ml_6h: max 3,000 mL (원본 집계 오류로 33,800 등 존재)
    - urine_ml_kg_hr_avg: max 10 mL/kg/hr
4. 출력 변수:
    - urine_ml_6h, urine_ml_kg_hr_avg: 클리핑 + 0 채움 (메인)
    - urine_*_original: 원본 보존 (결측 유지)
    - oliguria_flag: 보수적으로 1 채움
    - urine_missing_flag: 결측 여부 플래그

In [65]:
print("Step 7: Urine Output 결측치 처리 (업데이트 버전)")

# --- 7-1: 원본 보존 (결측 그대로 유지) ---
print("\n  7-1: 원본 결측 보존")

df['urine_ml_6h_original'] = df['urine_ml_6h'].copy()
df['urine_ml_kg_hr_avg_original'] = df['urine_ml_kg_hr_avg'].copy()
df['oliguria_flag_original'] = df['oliguria_flag'].copy()

original_missing = df['urine_ml_6h_original'].isna().sum()
print(f"    원본 결측: {original_missing:,}건 ({original_missing/len(df)*100:.1f}%)")

# --- 7-2: 결측 플래그 생성 ---
print("\n  7-2: 결측 플래그 생성")

df['urine_missing_flag'] = df['urine_ml_6h_original'].isna().astype(int)

missing_count = df['urine_missing_flag'].sum()
print(f"    urine_missing_flag=1 (결측): {missing_count:,}건 ({missing_count/len(df)*100:.1f}%)")
print(f"    urine_missing_flag=0 (관측): {len(df)-missing_count:,}건")

# --- 7-3: 이상치 클리핑 ---
# 원인: 06_urine_raw에서 시간당 중복 기록이 SUM되면서 비현실적 값 생성
# 임상 기준: 6시간 최대 3,000mL (극단적 다뇨도 커버)
print("\n  7-3: 이상치 클리핑")

URINE_6H_MAX = 3000          # mL
URINE_ML_KG_HR_MAX = 10      # mL/kg/hr

before_clip_6h = (df['urine_ml_6h'] > URINE_6H_MAX).sum()
before_clip_rate = (df['urine_ml_kg_hr_avg'] > URINE_ML_KG_HR_MAX).sum()

df['urine_ml_6h'] = df['urine_ml_6h'].clip(upper=URINE_6H_MAX)
df['urine_ml_kg_hr_avg'] = df['urine_ml_kg_hr_avg'].clip(upper=URINE_ML_KG_HR_MAX)

print(f"    urine_ml_6h: {before_clip_6h:,}건 클리핑 (max={URINE_6H_MAX})")
print(f"    urine_ml_kg_hr_avg: {before_clip_rate:,}건 클리핑 (max={URINE_ML_KG_HR_MAX})")

# --- 7-4: 결측 → 0 채움 (보수적 가정) ---
# 이유: 조기 악화 예측 목적 → false negative가 더 위험
# 기록 없음 = 소변 없었을 가능성 → 위험 신호로 가정
print("\n  7-4: 결측 → 0 채움 (보수적 가정)")

df['urine_ml_6h'] = df['urine_ml_6h'].fillna(0)
df['urine_ml_kg_hr_avg'] = df['urine_ml_kg_hr_avg'].fillna(0)

print(f"    urine_ml_6h: 결측 → 0")
print(f"    urine_ml_kg_hr_avg: 결측 → 0")

# --- 7-5: oliguria_flag 처리 ---
# 결측 → 1 (보수적: 모르면 핍뇨 있다고 가정)
print("\n  7-5: oliguria_flag 처리")

before_missing = df['oliguria_flag'].isna().sum()
df['oliguria_flag'] = df['oliguria_flag'].fillna(1)

print(f"    oliguria_flag: {before_missing:,}건 결측 → 1 (보수적)")

# --- 7-6: 최종 확인 ---
print("\n=== Urine 변수 최종 구성 ===")
print("  [메인 변수 - 클리핑 + 0 채움]")
print(f"    urine_ml_6h: 범위 0~{URINE_6H_MAX}, 결측 0건")
print(f"    urine_ml_kg_hr_avg: 범위 0~{URINE_ML_KG_HR_MAX}, 결측 0건")
print(f"    oliguria_flag: 결측 → 1 (보수적)")
print("  [보조 변수 - 원본 보존]")
print(f"    urine_*_original: 결측 {original_missing:,}건 보존")
print("  [플래그]")
print(f"    urine_missing_flag: 결측={missing_count:,}, 관측={len(df)-missing_count:,}")

print("\n✓ Urine Output 처리 완료")

Step 7: Urine Output 결측치 처리 (업데이트 버전)

  7-1: 원본 결측 보존
    원본 결측: 225,343건 (23.9%)

  7-2: 결측 플래그 생성
    urine_missing_flag=1 (결측): 225,343건 (23.9%)
    urine_missing_flag=0 (관측): 716,474건

  7-3: 이상치 클리핑
    urine_ml_6h: 1,054건 클리핑 (max=3000)
    urine_ml_kg_hr_avg: 18,994건 클리핑 (max=10)

  7-4: 결측 → 0 채움 (보수적 가정)
    urine_ml_6h: 결측 → 0
    urine_ml_kg_hr_avg: 결측 → 0

  7-5: oliguria_flag 처리
    oliguria_flag: 225,343건 결측 → 1 (보수적)

=== Urine 변수 최종 구성 ===
  [메인 변수 - 클리핑 + 0 채움]
    urine_ml_6h: 범위 0~3000, 결측 0건
    urine_ml_kg_hr_avg: 범위 0~10, 결측 0건
    oliguria_flag: 결측 → 1 (보수적)
  [보조 변수 - 원본 보존]
    urine_*_original: 결측 225,343건 보존
  [플래그]
    urine_missing_flag: 결측=225,343, 관측=716,474

✓ Urine Output 처리 완료


## Step 8: 제외 컬럼 삭제 및 최종 확인

* 제외 대상: sao2, ph, bilirubin (결측률 65~91%)

In [66]:
print("Step 8: 제외 컬럼 삭제 및 최종 확인")

# --- 8-1: 고결측률 컬럼 삭제 ---
# 제외 대상: sao2(91.5%), ph(68.4%), bilirubin(65.7%)
# 이유: 결측률 너무 높아 신뢰성 없음
drop_cols = ['sao2', 'ph', 'bilirubin']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')
print(f"  제외된 컬럼: {drop_cols}")

# --- 8-2: 피처 컬럼 정의 ---
# 메인 피처 (모델 입력용, 결측 없음)
main_features = [
    # Vital
    'hr', 'rr', 'spo2', 'temp', 'sbp', 'dbp', 'mbp',
    # Lab
    'lactate', 'creatinine', 'wbc', 'platelets', 'potassium', 'sodium',
    # GCS (FFill 적용)
    'gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total',
    # Urine (클리핑 + 0 채움)
    'urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag',
]

# 결측 플래그 (중증 신호 보존)
missing_flags = [
    'lactate_missing',
    'gcs_missing_flag',
    'urine_missing_flag',
]

# 원본 보존 (결측 유지, 분석용)
original_features = [
    'gcs_eye_original', 'gcs_verbal_original', 'gcs_motor_original', 'gcs_total_original',
    'urine_ml_6h_original', 'urine_ml_kg_hr_avg_original', 'oliguria_flag_original',
]

# --- 8-3: 메인 피처 결측 확인 ---
print("\n=== 메인 피처 결측 확인 ===")
total_missing = 0
for col in main_features + missing_flags:
    if col in df.columns:
        missing = df[col].isna().sum()
        total_missing += missing
        if missing > 0:
            print(f"  ⚠️ {col}: {missing:,} 결측")

if total_missing == 0:
    print("  ✓ 모든 메인 피처 결측 0건")

# --- 8-4: 원본 피처 결측 확인 (의도된 결측) ---
print("\n=== 원본 피처 결측 확인 (의도된 보존) ===")
for col in original_features:
    if col in df.columns:
        missing = df[col].isna().sum()
        print(f"  {col}: {missing:,}건 결측 (보존됨)")

# --- 8-5: 플래그 분포 확인 ---
print("\n=== Missing Flag 분포 ===")
for col in missing_flags:
    if col in df.columns:
        dist = df[col].value_counts().to_dict()
        print(f"  {col}: {dist}")

# --- 8-6: 전체 컬럼 요약 ---
print("\n=== 전체 컬럼 구성 ===")
print(f"  메인 피처: {len(main_features)}개")
print(f"  결측 플래그: {len(missing_flags)}개")
print(f"  원본 보존: {len(original_features)}개")
print(f"  총 컬럼 수: {len(df.columns)}개")

print("\n✓ Step 8 완료")

Step 8: 제외 컬럼 삭제 및 최종 확인
  제외된 컬럼: ['sao2', 'ph', 'bilirubin']

=== 메인 피처 결측 확인 ===
  ✓ 모든 메인 피처 결측 0건

=== 원본 피처 결측 확인 (의도된 보존) ===
  gcs_eye_original: 89,650건 결측 (보존됨)
  gcs_verbal_original: 89,841건 결측 (보존됨)
  gcs_motor_original: 91,630건 결측 (보존됨)
  gcs_total_original: 93,615건 결측 (보존됨)
  urine_ml_6h_original: 225,343건 결측 (보존됨)
  urine_ml_kg_hr_avg_original: 225,343건 결측 (보존됨)
  oliguria_flag_original: 225,343건 결측 (보존됨)

=== Missing Flag 분포 ===
  lactate_missing: {1: 674429, 0: 267388}
  gcs_missing_flag: {0: 848202, 1: 93615}
  urine_missing_flag: {0: 716474, 1: 225343}

=== 전체 컬럼 구성 ===
  메인 피처: 20개
  결측 플래그: 3개
  원본 보존: 7개
  총 컬럼 수: 57개

✓ Step 8 완료


In [67]:
print("\n" + "="*60)
print("전처리 완료 요약")
print("="*60)

# --- 피처 그룹 정의 ---
vital_features = ['hr', 'rr', 'spo2', 'temp', 'sbp', 'dbp', 'mbp']
lab_features = ['lactate', 'creatinine', 'wbc', 'platelets', 'potassium', 'sodium']
gcs_features = ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']
urine_features = ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag']
missing_flags = ['lactate_missing', 'gcs_missing_flag', 'urine_missing_flag']
original_features = [
    'gcs_eye_original', 'gcs_verbal_original', 'gcs_motor_original', 'gcs_total_original',
    'urine_ml_6h_original', 'urine_ml_kg_hr_avg_original', 'oliguria_flag_original'
]

# 메인 피처 (모델 입력용)
main_features = vital_features + lab_features + gcs_features + urine_features

# --- 메인 피처 결측 확인 ---
print("\n=== 메인 피처 결측 확인 ===")
main_missing = df[main_features].isna().sum().sum()
print(f"총 결측: {main_missing}개")
if main_missing == 0:
    print("✓ 모든 메인 피처 결측 0건 - 모델 입력 준비 완료")

# --- 피처 그룹별 기술 통계 ---
print("\n=== Vital Signs 기술 통계 ===")
print(df[vital_features].describe().round(2))

print("\n=== Lab Values 기술 통계 ===")
print(df[lab_features].describe().round(2))

print("\n=== GCS 기술 통계 ===")
print(df[gcs_features].describe().round(2))

print("\n=== Urine 기술 통계 ===")
print(df[urine_features].describe().round(2))

# --- Missing Flag 분포 ---
print("\n=== Missing Flag 분포 ===")
for col in missing_flags:
    if col in df.columns:
        total = len(df)
        missing_cnt = df[col].sum()
        print(f"  {col}: 결측={missing_cnt:,} ({missing_cnt/total*100:.1f}%), 관측={total-missing_cnt:,} ({(total-missing_cnt)/total*100:.1f}%)")

# --- 데이터 규모 ---
print("\n=== 데이터 규모 ===")
print(f"  총 행 수: {len(df):,}")
print(f"  고유 환자 수: {df['stay_id'].nunique():,}")
print(f"  메인 피처 수: {len(main_features)}개")
print(f"  결측 플래그 수: {len(missing_flags)}개")
print(f"  원본 보존 피처 수: {len(original_features)}개")
print(f"  총 컬럼 수: {len(df.columns)}개")


전처리 완료 요약

=== 메인 피처 결측 확인 ===
총 결측: 0개
✓ 모든 메인 피처 결측 0건 - 모델 입력 준비 완료

=== Vital Signs 기술 통계 ===
              hr         rr       spo2       temp        sbp        dbp  \
count  941817.00  941817.00  941817.00  941817.00  941817.00  941817.00   
mean       83.00      19.30      96.14      36.86     122.78      67.22   
std        17.22       4.32       2.27       0.44      18.78      13.23   
min        25.50       4.00      51.50      30.11      41.00      21.00   
25%        70.50      16.33      94.83      36.61     109.00      58.00   
50%        81.33      18.67      96.25      36.83     121.50      66.20   
75%        93.83      21.67      97.67      37.06     135.00      75.60   
max       188.50      58.00     100.00      41.11     238.83     178.00   

             mbp  
count  941817.00  
mean       82.53  
std        13.36  
min        30.00  
25%        73.00  
50%        81.33  
75%        91.00  
max       182.00  

=== Lab Values 기술 통계 ===
         lactate  creatinine

## Step 9: 저장

In [68]:
output_path = os.path.join(OUTPUT_DIR, 'preprocessed.csv')
df.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: preprocessed.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df):,}개")
print(f"  - 컬럼 수: {len(df.columns)}개")
print(f"  - 경로: {output_path}")


✓ 저장 완료: preprocessed.csv
  - 파일 크기: 313.07 MB
  - 행 수: 941,817개
  - 컬럼 수: 57개
  - 경로: ../data/processed/preprocessed.csv


In [69]:
# --- 저장된 컬럼 목록 ---
print("\n=== 저장된 컬럼 목록 ===")
print("\n[ID/시간]")
id_cols = ['stay_id', 'subject_id', 'hadm_id', 'observation_hour', 'observation_start', 'observation_end']
for col in id_cols:
    if col in df.columns:
        print(f"  - {col}")

print("\n[메인 피처 - Vital]")
for col in vital_features:
    print(f"  - {col}")

print("\n[메인 피처 - Lab]")
for col in lab_features:
    print(f"  - {col}")

print("\n[메인 피처 - GCS]")
for col in gcs_features:
    print(f"  - {col}")

print("\n[메인 피처 - Urine]")
for col in urine_features:
    print(f"  - {col}")

print("\n[결측 플래그]")
for col in missing_flags:
    if col in df.columns:
        print(f"  - {col}")

print("\n[원본 보존 (결측 유지)]")
for col in original_features:
    if col in df.columns:
        print(f"  - {col}")

print("\n[레이블]")
label_cols = [col for col in df.columns if 'next_' in col]
for col in label_cols:
    print(f"  - {col}")


=== 저장된 컬럼 목록 ===

[ID/시간]
  - stay_id
  - subject_id
  - hadm_id
  - observation_hour
  - observation_start
  - observation_end

[메인 피처 - Vital]
  - hr
  - rr
  - spo2
  - temp
  - sbp
  - dbp
  - mbp

[메인 피처 - Lab]
  - lactate
  - creatinine
  - wbc
  - platelets
  - potassium
  - sodium

[메인 피처 - GCS]
  - gcs_eye
  - gcs_verbal
  - gcs_motor
  - gcs_total

[메인 피처 - Urine]
  - urine_ml_6h
  - urine_ml_kg_hr_avg
  - oliguria_flag

[결측 플래그]
  - lactate_missing
  - gcs_missing_flag
  - urine_missing_flag

[원본 보존 (결측 유지)]
  - gcs_eye_original
  - gcs_verbal_original
  - gcs_motor_original
  - gcs_total_original
  - urine_ml_6h_original
  - urine_ml_kg_hr_avg_original
  - oliguria_flag_original

[레이블]
  - death_next_6h
  - vent_next_6h
  - pressor_next_6h
  - composite_next_6h
  - death_next_12h
  - vent_next_12h
  - pressor_next_12h
  - composite_next_12h
  - death_next_24h
  - vent_next_24h
  - pressor_next_24h
  - composite_next_24h


In [70]:
print("\n=== 09. Preprocessing 완료 ===")


=== 09. Preprocessing 완료 ===


---

## 1. 레이블 분포

In [71]:
print("=== 레이블 분포 ===")
label_cols = [col for col in df.columns if 'next_' in col]

for col in label_cols:
    if col in df.columns:
        counts = df[col].value_counts()
        ratio = df[col].mean() * 100
        print(f"{col}: 1={counts.get(1, 0):,} ({ratio:.2f}%), 0={counts.get(0, 0):,}")

=== 레이블 분포 ===
death_next_6h: 1=1,745 (0.19%), 0=940,072
vent_next_6h: 1=8,995 (0.96%), 0=932,822
pressor_next_6h: 1=4,527 (0.48%), 0=937,290
composite_next_6h: 1=13,097 (1.39%), 0=928,720
death_next_12h: 1=3,791 (0.40%), 0=938,026
vent_next_12h: 1=15,473 (1.64%), 0=926,344
pressor_next_12h: 1=8,127 (0.86%), 0=933,690
composite_next_12h: 1=22,910 (2.43%), 0=918,907
death_next_24h: 1=8,670 (0.92%), 0=933,147
vent_next_24h: 1=24,286 (2.58%), 0=917,531
pressor_next_24h: 1=13,289 (1.41%), 0=928,528
composite_next_24h: 1=37,804 (4.01%), 0=904,013


## 2. 피처 기술통계

In [72]:
# 주요 피처 분포 확인
print("\n=== 피처 기술통계 ===")
feature_cols = [
    'hr', 'rr', 'spo2', 'temp', 'sbp', 'dbp', 'mbp',
    'lactate', 'creatinine', 'wbc', 'platelets', 'potassium', 'sodium',
    'gcs_total', 'urine_ml_6h', 'urine_ml_kg_hr_avg'
]
df[feature_cols].describe().round(2)


=== 피처 기술통계 ===


,hr,rr,spo2,temp,sbp,dbp,mbp,lactate,creatinine,wbc,platelets,potassium,sodium,gcs_total,urine_ml_6h,urine_ml_kg_hr_avg
count,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00,941817.00
mean,83.00,19.30,96.14,36.86,122.78,67.22,82.53,1.52,1.27,11.00,206.08,4.08,138.41,13.84,386.45,2.15
std,17.22,4.32,2.27,0.44,18.78,13.23,13.36,0.86,1.36,6.15,95.50,0.54,4.93,2.51,413.17,2.38
min,25.50,4.00,51.50,30.11,41.00,21.00,30.00,0.20,0.10,0.10,5.00,1.50,110.00,3.00,0.00,0.00
25%,70.50,16.33,94.83,36.61,109.00,58.00,73.00,1.30,0.70,7.50,148.00,3.70,136.00,14.00,35.00,0.18
50%,81.33,18.67,96.25,36.83,121.50,66.20,81.33,1.30,0.90,9.90,195.00,4.00,139.00,15.00,300.00,1.40
75%,93.83,21.67,97.67,37.06,135.00,75.60,91.00,1.30,1.20,13.00,249.00,4.30,141.00,15.00,550.00,3.19
max,188.50,58.00,100.00,41.11,238.83,178.00,182.00,24.00,24.70,98.60,998.00,10.00,170.00,15.00,3000.00,10.00


## 3. 환자 수 및 윈도우 분포

In [73]:
print("\n=== 데이터 규모 ===")
print(f"총 행 수: {len(df):,}")
print(f"고유 환자 수: {df['stay_id'].nunique():,}")
print(f"환자당 평균 윈도우: {len(df) / df['stay_id'].nunique():.1f}개")

# 환자당 윈도우 수 분포
windows_per_patient = df.groupby('stay_id').size()
print(f"\n환자당 윈도우 수:")
print(f"  min: {windows_per_patient.min()}, max: {windows_per_patient.max()}")
print(f"  median: {windows_per_patient.median():.0f}")


=== 데이터 규모 ===
총 행 수: 941,817
고유 환자 수: 23,938
환자당 평균 윈도우: 39.3개

환자당 윈도우 수:
  min: 1, max: 67
  median: 38


## 4. 범주형 변수 현황

In [74]:
print("\n=== 범주형 변수 ===")
if 'gender' in df.columns:
    print(f"gender: {df['gender'].value_counts().to_dict()}")
if 'first_careunit' in df.columns:
    print(f"first_careunit:\n{df['first_careunit'].value_counts()}")


=== 범주형 변수 ===
gender: {'M': 500184, 'F': 441633}
first_careunit:
first_careunit
Neuro Intermediate                                  167545
Medical/Surgical Intensive Care Unit (MICU/SICU)    151670
Surgical Intensive Care Unit (SICU)                 149952
Medical Intensive Care Unit (MICU)                  147921
Coronary Care Unit (CCU)                            109013
Trauma SICU (TSICU)                                  79387
Cardiac Vascular Intensive Care Unit (CVICU)         71868
Neuro Stepdown                                       32588
Neuro Surgical Intensive Care Unit (Neuro SICU)      28675
PACU                                                  1651
Surgery/Vascular/Intermediate                         1157
Intensive Care Unit (ICU)                              327
Surgery/Trauma                                          39
Medicine                                                12
Neurology                                               12
Name: count, dtype: int64


## 5. Missing Indicator 분포

In [75]:
print("\n=== Missing Indicator ===")
if 'lactate_missing' in df.columns:
    print(f"lactate_missing: {df['lactate_missing'].value_counts().to_dict()}")
if 'urine_missing' in df.columns:
    print(f"urine_missing: {df['urine_missing'].value_counts().to_dict()}")


=== Missing Indicator ===
lactate_missing: {1: 674429, 0: 267388}


## 6. 실제 측정된 lactate만 확인 (imputed 제외)

In [76]:
# 실제 측정된 lactate만 확인 (imputed 제외)
print("=== Lactate 실제 측정값 분포 ===")
lactate_measured = df[df['lactate_missing'] == 0]['lactate']
print(lactate_measured.describe())

=== Lactate 실제 측정값 분포 ===
count    267388.000000
mean          1.878714
std           1.488141
min           0.200000
25%           1.100000
50%           1.500000
75%           2.100000
max          24.000000
Name: lactate, dtype: float64
